# Reversión de tablas Medallion

La celda muestra el tipo y la ubicación de las seis tablas del proyecto. Con `drop_tables = False` se ejecuta la revisión. Para eliminar las tablas administradas se cambia el valor a `True` y se ejecuta la celda. La eliminación sigue el orden Gold, Silver y Bronze.


In [ ]:
catalog = "catalog_smartdata"
drop_tables = False

project_tables = [
    "bitcoin_gold.sentiment_summary",
    "bitcoin_gold.bitcoin_daily_metrics",
    "bitcoin_silver.bitcoin_daily",
    "bitcoin_silver.binance_minutes",
    "bitcoin_bronze.fear_greed_index",
    "bitcoin_bronze.binance_prices",
]

inventory = []
for table in project_tables:
    name = f"{catalog}.{table}"
    if not spark.catalog.tableExists(name):
        inventory.append((name, "AUSENTE", ""))
        continue

    description = spark.sql(
        "DESCRIBE TABLE EXTENDED IDENTIFIER(:name)",
        args={"name": name},
    )
    details = {row.col_name: row.data_type for row in description.collect()}
    inventory.append((name, details.get("Type", "DESCONOCIDO"), details.get("Location", "")))

display(spark.createDataFrame(inventory, ["tabla", "tipo", "ubicacion"]))

if drop_tables:
    unexpected = [name for name, kind, _ in inventory if kind not in ("MANAGED", "AUSENTE")]
    if unexpected:
        raise ValueError(f"Se encontraron tablas no administradas: {', '.join(unexpected)}")

    for name, kind, _ in inventory:
        if kind == "MANAGED":
            spark.sql("DROP TABLE IF EXISTS IDENTIFIER(:name)", args={"name": name})
            print(f"Eliminada: {name}")
